$$\Large\boxed{\text{AME 5202 Deep Learning, Even Semester 2026}}$$

$$\large\text{Theme}: \underline{\text{building a softmax classifier from scratch using efficient computations in PyTorch}}$$

---

Load libraries

---

In [1]:
## Load libraries
import numpy as np
import sys
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

RuntimeError: operator torchvision::nms does not exist

---

The **MNIST (Modified National Institute of Standards and Technology) dataset** is one of the most widely used benchmark datasets in machine learning and computer vision.

It contains:

- **70,000 grayscale images**
- **28 × 28 pixels** per image
- **10 classes (digits 0–9)**
- Split into **60,000 training** and **10,000 test** images

Each image shows a centered handwritten digit as in the example below:

![mnist](https://1drv.ms/i/c/37720f927b6ddc34/IQQ6UqroXhRjSp1y9MW00bT_AR8-ENNoQV9lkr8uUoFpf7Q)

We will load the MNIST dataset and transform the images (normalize pixel values from 0-1 and flatten) using PyTorch.


---

In [ ]:
# Define PyTorch transformation pipeline for the MNIST dataset
transform = transforms.Compose([
    # Converts [28x28] gradyscale image → [1,28,28] tensor normalized to [0, 1]
    transforms.ToTensor(),
    # Flatten to [784]
    transforms.Lambda(lambda x: torch.flatten(x))
])

# Load train dataset
train_dataset = datasets.MNIST(
    root = './data',
    train = True,
    download = True,
    transform = transform
)

# Load test dataset
test_dataset = datasets.MNIST(
    root = './data',
    train = False,
    download = True,
    transform = transform
)

num_samples = len(train_dataset), len(test_dataset)
num_features = train_dataset[0][0].shape[0]
num_labels = torch.unique(train_dataset.targets).numel()

print('MNIST set')
print('---------------------')
print(f'Number of training samples = {num_samples[0]}, test samples = {num_samples[1]}')
print(f'Number of features = {num_features}')
print(f'Number of output labels = {num_labels}')


MNIST set
---------------------
Number of training samples = 60000, test samples = 10000
Number of features = 784
Number of output labels = 10


---

A generic layer class with forward and backward methods

----

In [ ]:
class Layer():
  def __init__(self):
    self.input = None
    self.output = None

  def forward(self, input):
    raise NotImplementedError("Forward propagation not implemented")

  def backward(self, output_gradient, learning_rate):
    raise NotImplementedError("Backward propagation not implemented")

---

$${\color{red}{\textbf{Complete the ? in the code cell below:}}}$$

Categorical crossentropy (CCE) loss and its gradient for the batch samples.

---

In [ ]:
## Define the loss function and its gradient
class CategoricalCrossEntropyLoss:
  def __init__(self, num_labels):
    self.num_labels = num_labels
    self.Y_onehot = None

  def forward(self, Y, Yhat):
    self.Y_onehot = F.one_hot(Y, num_classes = self.num_labels).float()
    return torch.mean(-torch.log(torch.sum(self.Y_onehot*Yhat, dim = -1)))

  def backward(self, Y, Yhat):
    return -self.Y_onehot/Yhat

---

$${\color{red}{\textbf{Complete the ? in the code cell below:}}}$$

Softmax activation layer class


---

In [ ]:
## Softmax activation layer class
class Softmax(Layer):
  def __init__(self):
    super(Softmax, self).__init__()

  def forward(self, input):
    self.input = input
    # Calculate softmax(input)
    self.output = F.softmax(self.input)
    return self.output

  def backward(self, output_gradient, learning_gradient = None):
    I = torch.eye(self.output.shape[1])
    softmax_local_gradient = (I - self.output.unsqueeze(-1)) * (self.output.unsqueeze(-2))
    # Calculate gradient flowing back on the input side of the softmax layer
    input_gradient = torch.einsum('ij,ijk->ik', output_gradient, softmax_local_gradient)
    return input_gradient

---

$${\color{red}{\textbf{Complete the ? in the code cell below:}}}$$

Dense layer class

---

In [ ]:
## Dense layer class
class Dense(Layer):
  def __init__(self, input_size, output_size, reg_strength = 0.0):
    super(Dense, self).__init__()
    self.weights = torch.randn(input_size+1, output_size, requires_grad = True) * 0.01
    with torch.no_grad():
      self.weights[-1].fill_(0.01) # set all bias values equal to 0.01
    self.reg_strength = reg_strength
    self.reg_loss = None

  def forward(self, input):
    self.input = torch.hstack((input, torch.ones(input.shape[0], 1))) # adding the bias feature as the last column
    self.output= self.input @ self.weights
    self.reg_loss = self.reg_strength * torch.sum(self.weights[:-1] * self.weights[:-1])
    return self.output

  def backward(self, output_gradient, learning_rate = 1e-02):
    # Calculate local gradient w.r.t. weights
    weights_gradient = (1/output_gradient.shape[0]) * (torch.einsum('ij, ik -> jk', self.input, output_gradient))
    # Add the Regularization Gradient
    weights_gradient += self.reg_strength * 2 * torch.vstack([self.weights[:-1],torch.zeros(1,self.weights.shape[1])])
    # Update weights for dense layer
    with torch.no_grad():
      self.weights -= weights_gradient * learning_rate

---

$${\color{red}{\textbf{Complete the ? in the code cell below:}}}$$


The neural network class for the softmax classifier.


---

In [ ]:
class NeuralNetwork:
  def __init__(self,
               num_features,
               num_labels,
               loss_fn,
               learning_rate = 1e-02,
               reg_strength = 0.0):
    self.num_features = num_features
    self.num_labels = num_labels
    self.loss_fn = loss_fn
    self.learning_rate = learning_rate
    self.reg_strength = reg_strength
    self.reg_loss = None
    # Architecture
    self.layers = [
        Dense(self.num_features, self.num_labels),
        Softmax()
        ]

  # Forward propagation
  def forward(self, x):
    self.reg_loss = 0.0
    for layer in self.layers:
        if hasattr(layer, 'weights'):
            x, reg_loss = layer.forward(x)
            self.reg_loss += reg_loss
        else:
            x = layer.forward(x)
    return x

  # Backward propagation
  def backward(self, gradient):
    for layer in reversed(self.layers):
      gradient = layer.backward(gradient, self.learning_rate)

---

$${\color{red}{\textbf{Complete the ? in the code cell below:}}}$$

Train the softmax classifier.


---

In [ ]:
# Initialize model and optimizer
learning_rate = 1e-01
nepochs = 100
batch_size = 100
reg_strength = 0.1

# Choose appropriate loss function
loss_fn = CategoricalCrossEntropyLoss(num_labels)

# Data loader for batch processing
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

model = NeuralNetwork(num_features, num_labels, loss_fn, learning_rate, reg_strength)

# Create empty list to store training and test losses over each epoch
train_loss = [None]*nepochs
test_loss = [None]*nepochs

for epoch in range(nepochs):
  loss = 0.0

  for x_batch, y_batch in train_loader:
    # Forward pass
    predictions = model.forward(x_batch)
    loss = loss + model.loss_fn.forward(y_batch, predictions) + model.reg_loss

    # Backward pass
    loss_gradient = model.loss_fn.backward(y_batch, predictions)
    model.backward(loss_gradient)

  train_loss[epoch] = loss.detach().numpy() / len(train_loader)

  # Test loss calculation
  loss = 0.0

  with torch.no_grad():
    for x_batch, y_batch in test_loader:
      predictions = model.forward(x_batch)
      loss = loss + model.loss_fn.forward(y_batch, predictions) + model.reg_loss

  test_loss[epoch] = loss.detach().numpy() / len(test_loader)

  print(f"Epoch {epoch + 1}/{nepochs}, Train Loss: {train_loss[epoch]:.4f}, Test Loss: {test_loss[epoch]:.4f}")

ValueError: too many values to unpack (expected 2)

---

Plot training and test loss vs. epoch

---

In [ ]:
## Plot train and test loss as a function of epoch:
fig, ax = plt.subplots(1, 1, figsize = (4, 4))
fig.tight_layout(pad = 4.0)
ax.plot(train_loss, 'b', label = 'Train')
ax.plot(test_loss, 'r', label = 'Test')
ax.set_xlabel('Epoch', fontsize = 12)
ax.set_ylabel('Loss value', fontsize = 12)
ax.legend()
ax.set_title('Loss vs. Epoch', fontsize = 14);

---

Assess model performance on test data

---

In [ ]:
all_preds = []
all_true = []

with torch.no_grad():
  for x_batch, y_batch in test_loader:
    outputs = model.forward(x_batch)
    preds = torch.argmax(outputs, dim = 1)
    all_preds.append(preds)
    all_true.append(y_batch)

ypred = torch.cat(all_preds)
ytrue = torch.cat(all_true)

accuracy = (ypred == ytrue).float().mean() * 100
print(f'Accuracy on test data = {accuracy:3.2f}')

print(f'Confusion matrix:\n{confusion_matrix(ytrue.cpu(), ypred.cpu())}')

---

Plot a random test sample with its predicted label printed above the plot.

---

In [ ]:
## Plot a random test sample with its predicted label printed above the plot
test_index = np.random.choice(len(test_dataset))
fig, ax = plt.subplots(1, 1, figsize = (2, 2))
print(
    f"Image {'classified correctly' if ypred[test_index] == ytrue[test_index] else 'classified incorrectly'} "
    f"as {ypred[test_index]}"
)
ax.imshow(test_dataset[test_index][0].detach().numpy().reshape(28, 28), cmap = 'gray');